# Leon Levy Curation Script

This script is the latest version of the batch process workflow that deposits data files from the Leon Levy Expedition to Ashkelon into Harvard Dataverse. The full collection is available here: <add link>. 

This script relies on functions stored in the utils.py file. 

Basic document outline:
1. Establish metadata file for batch. Eg. for Photos subcollection, datasets are split into year; for Illustrations, datasets are split into arbitrary buckets of 500 files.
2. Deposit datasets using EasyDataverse with built in python DVUploader.
3. Manually QA Datasets on Harvard Dataverse
4. Finialize & publish datasets back in EasyDataverse.

# 0. Setup

Bypass SSL verification because of HUIT rate limiting issues...

This patch only works if you're connected to the Harvard VPN. Otherwise you're subject to the HUIT rate limiter that they have deployed to manage bot traffic. 

In [1]:
# === PATCH CELL: Run this FIRST ===

import httpx

# Save originals in case you want to revert
_OriginalClient = httpx.Client
_OriginalAsyncClient = httpx.AsyncClient
_OriginalGet = httpx.get

# Patch Client (sync)
class InsecureClient(httpx.Client):
    def __init__(self, *args, **kwargs):
        kwargs["verify"] = False
        super().__init__(*args, **kwargs)

# Patch AsyncClient
class InsecureAsyncClient(httpx.AsyncClient):
    def __init__(self, *args, **kwargs):
        kwargs["verify"] = False
        super().__init__(*args, **kwargs)

# Patch httpx.get
def insecure_get(*args, **kwargs):
    kwargs["verify"] = False
    return _OriginalGet(*args, **kwargs)

# Apply all patches
httpx.Client = InsecureClient
httpx.AsyncClient = InsecureAsyncClient
httpx.get = insecure_get


# Patch requests (used by pyDataverse)
import requests

_original_requests_request = requests.Session.request

def insecure_requests_request(self, method, url, **kwargs):
    kwargs["verify"] = False
    return _original_requests_request(self, method, url, **kwargs)

requests.Session.request = insecure_requests_request


In [2]:
# === Raise rate limits for large deposits ===

import resource

# Check current limits
soft, hard = resource.getrlimit(resource.RLIMIT_NOFILE)
print(f"Current limits — soft: {soft}, hard: {hard}")

# Raise the soft limit to the hard limit ceiling
resource.setrlimit(resource.RLIMIT_NOFILE, (hard, hard))

soft, hard = resource.getrlimit(resource.RLIMIT_NOFILE)
print(f"Updated limits — soft: {soft}, hard: {hard}")

Current limits — soft: 4096, hard: 9223372036854775807
Updated limits — soft: 9223372036854775807, hard: 9223372036854775807


#### Global Variables

In [3]:
# set curation source path
source_path = '/Users/katherinemika/Desktop/curation/leon_levy/summer2026'

# path to inventory file
g_dataverse_inventory_file = '/Users/katherinemika/Desktop/curation/leon_levy/summer2026/metadata_fieldbooks_more.csv'

# dataset names
g_dataset_names = []

# dataset inventories (keyed on dataset name)
g_dataset_inventories = {}

# dataset metadata (keyed on dataset name)
g_dataset_metadata = {}

# dataverse installation + API key -- loaded from .env, never hard-coded.
# See .env.example. DATAVERSE_TARGET picks the installation: "demo" or "harvard".
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)

_prefix = "DEMO_" if os.environ["DATAVERSE_TARGET"] == "demo" else ""
dv_installation_url = os.environ[f"{_prefix}DATAVERSE_URL"]
dv_api_key = os.environ[f"{_prefix}DATAVERSE_API_TOKEN"]

# Previously pointed at the staging host below, which requires the Harvard VPN.
# To use it again, set DATAVERSE_URL in .env:
#   https://dvn-cloud-app-2.lib.harvard.edu
print(f"Dataverse target: {os.environ['DATAVERSE_TARGET']} -> {dv_installation_url}")

# dataverse collection name
dv_collection = 'ashkelonexcavations'

# dataverse inventory dataframe
dv_inventory_df = None

# full path to location of datafiles (e.g., ../data/trade_statistics)
#datafiles_path = '/Users/katherinemika/Library/CloudStorage/OneDrive-HarvardUniversity/LeonLevy' #separated into subcollection and years
datafiles_path = '/Users/katherinemika/Desktop/curation/leon_levy/summer2026/fieldbooksandMC'
# dataverse dataset information (keyed on dataset name)
dv_dataset_info = {}

# datafile metadata (dataframe of datafile metadata, keyed on dataset name)
g_datafile_metadata = {}

#### Hard Coded Metadata Fields
These fields can be set at the script level, because they will apply to all datasets in this collection. 

In [4]:
# dataset authors
author1_name = 'Master, Daniel M.'
author1_affil = "Wheaton College"
author2_name = 'Stager, Lawrence E.'
author2_affil = 'Harvard University'

# dataset contact information
dataset_contact = 'Master, Daniel M.'
dataset_contact_email = 'daniel.master@wheaton.edu'

# dataset subject
subj = 'Arts and Humanities'

# dataset keyword
kws = 'Archaeology'

# language 
lang = "English"

# Funding info
funding_agency = 'The Leon Levy Foundation'

# Depositor 
depositor = "Mika, Katherine"

# Geospatial
geo_coverage = "Ashkelon"
geo_unit = "City"

In [5]:
# import libraries
import json
import requests
import pprint as pprint
import pandas as pd
import rich
import os
import sys
import numpy as np
from pyDataverse.models import Dataset
import requests
import dvuploader as dv
import unicodedata
import time
if source_path not in sys.path:
    sys.path.append(source_path)

In [6]:
# Local functions
def convert_year(two_digit_year):
    if pd.isna(two_digit_year):  # check if the value is NaN
        return np.nan
    year = int(two_digit_year)
    if year <= 49:
        return 2000 + year
    else:
        return 1900 + year

def create_dataset_metadata(author_name1,
                            author_name2,
                            affiliation1,
                            affiliation2,
                            contactName,
                            contactEmail,
                            subject,
                            keywords,
                            lang,
                            funding,
                            depositor,
                            geo_coverage,
                            geo_unit,
                            title, 
                            inventory):
    """
    Create a dictionary of dataset metadata

    Parameters
    ----------
    author : str
        Dataset author names, assigned in Step 0 of curation script.
    affiliation : str
        Dataset athor affiliations, assigned in Step 0 of curation script.
    contact : str
        Dataset contact name (may be same as author), assigned in Step 0 of curation script.
    email : str
        Dataset contact email address, assigned in Step 0 of curation script.
    subject : str
        Dataset subjects, separated by semicolons, assigned in Step 0 of curation script.
    keywords : str
        Dataset keywords, separated by semicolons, assigned in Step 0 of curation script.
    lang : str
        Dataset language, assigned in Step 0 of curation script.
    funding : str
        Dataset funder information, assigned in Step 0 of curation script.
    depositer : str
        Person who deposited dataset into Dataverse
    geo_coverage : str
        Location, in natural language, of where data was collected or refers to
    geo_unit : str
        Level of geographic aggregation for dataset
    title : str
        Name of dataset (e.g., Tonnage: 1)
    inventory : DataFrame
        DataFrame containing file metadata

    Return
    ------
    dict
    """
    # Validate parameters
    if ((not author_name1) or
        (not affiliation1) or
        (not contactName) or
        (not contactEmail) or
        (not title) or 
        (inventory.empty)):
        print('Error: One or more invalid parameter values')
        return {}
    
    # Check the inventory for required fields
    required_fields = ['dataset_title', 'dataset_description']
    for field in required_fields:
        if field not in inventory.columns:
            print(f'Error: Missing required field {field} in inventory')
            return {}

    # Get the index of the first file in the inventory
    index = inventory.index.values[0]
    
    # Collect metadata variables
    dataset_title = inventory.at[index, 'dataset_title']
    description = inventory.at[index, 'dataset_description']
    
    # Process keywords
    keyword_str = keywords
    keywords = keyword_str.split(';') if isinstance(keyword_str, str) else []
    kws = [{'keywordValue': kw} for kw in keywords]

    #make sure language is a list
    language_value = lang
    language_value = [language_value]    

    #prod_date
    #production_date = inventory.at[index, 'prod_date']

    # Build the dataset metadata dictionary
    dataset_metadata = {
        'title': dataset_title,
        'author': [{'authorName': author_name1, 'authorAffiliation': affiliation1},
                   {'authorName': author_name2, 'authorAffiliation': affiliation2}],
        'description': [{'dsDescriptionValue': description}],
        'contact': [{'datasetContactName': contactName, 'datasetContactAffiliation': affiliation1, 'datasetContactEmail': contactEmail}],
        'subject': [subject],
        'license': 'CC BY-NC-ND 4.0',
        'keywords': kws,
        'funding': funding,
        #'productionDate': production_date,
        'depositor': depositor,
        'language': language_value,
        'geographicCoverage': geo_coverage,
        'geographicUnit': geo_unit
    }

    return dataset_metadata


def create_datafile_metadata(inventory_df):
    """
    Create metadata for data files based on a template

    Parameters
    -----------
    inventory_df : DataFrame
        DataFrame containing list of data files to upload

    Return
    -------
    DataFrame
    
    """
    # validate params
    if (inventory_df.empty == True):
        print('Error: One or more invalad parameters')
        return pd.DataFrame

    # check DataFrame for required fields
    if ((not 'filename' in inventory_df.columns) or
        (not 'dataset_title' in inventory_df.columns)):
        print('Error: One or more missing required fields in inventory')
        return pd.DataFrame()

    ## prepare series of values to add to metadata df

    # prepare file names & descriptions for actual file
    all_filenames = []
    all_descriptions = []

    # iterate through inventory and create datafile metadata
    for index, row in inventory_df.iterrows():
        # get inventory variables
        filename = row['filename']
        all_filenames.append(filename)
        dataset = row['dataset_title']
        df_description = row['file_description']
        all_descriptions.append(df_description)


    # build DataFrame
    df = pd.DataFrame({
        'filename': all_filenames,
        'df_description': all_descriptions
    })

    return df


def create_dataset(ds, dataset_metadata):
    """
    Create a dataverse dataset using easyDataverse.
    Note that metadata fields are hardcoded to reflect dataset's requirements. 

    Parameters
    ----------
    ds : initialized easyDataverse Dataset
    dataset_metadata : dict
        Dictionary of dataset metadata values

    Return
    ------
    dict: 
        {status: bool, dataset_id: int, dataset_pid: str}

    """

    # validate parameters
    if ((not ds) or
        (not dataset_metadata)):
        return {
            'status':False, 
            'dataset_id':-1, 
            'dataset_pid':''
        }


    # populate the dataset model with metadata values
    ds.citation.title = dataset_metadata.get('title')

    for authors in dataset_metadata.get('author'):
        ds.citation.add_author(name = authors['authorName'],
                              affiliation = authors['authorAffiliation'])

    for desc in dataset_metadata.get('description'):
        ds.citation.add_ds_description(value=desc['dsDescriptionValue'])
    
    for contact in dataset_metadata.get('contact'):
        ds.citation.add_dataset_contact(name = contact['datasetContactName'],
                                        email = contact['datasetContactEmail'])

    ds.citation.subject = dataset_metadata.get('subject')

    for keyword in dataset_metadata.get('keywords'):
        ds.citation.add_keyword(value = keyword['keywordValue'])

    ds.citation.depositor = dataset_metadata.get('depositor')
    ds.citation.language = dataset_metadata.get('language')
    ds.citation.add_grant_number(agency = dataset_metadata.get('funding'))
    
    #production_date = str(dataset_metadata.get('productionDate'))
    #ds.citation.production_date = production_date


    ds.geospatial.unit.append(dataset_metadata.get('geographicUnit'))
    #ds.geospatial.unit = dataset_metadata.get('geographicUnit')
    ds.geospatial.add_coverage(other_geographic_coverage = dataset_metadata.get('geographicCoverage'))

    return ds



def add_files_to_dataset(dataset, file_inventory, datafiles_path):
    """
    Add files to a dataset in easyDataverse

    Parameters
    ----------
    
    dataset : easyDataverse initialized dataset
    file_inventory : DataFrame
        DataFrame of file metadata, keyed on dataset
    datafiles_path : global variable
        Path to file directory set in global var cell
    
    Return
    ------
    dict: 
        {status: bool, dataset_id: int, dataset_pid: str}

    """
    # check inventoryDataFrame for required fields
    if ((not 'filename' in file_inventory.columns) or
        (not 'df_description' in file_inventory.columns)):
        print('Error: One or more missing required fields in inventory')
        return pd.DataFrame()
        
    for index, row in file_inventory.iterrows():
        dataset.add_file(
            local_path = os.path.join(datafiles_path, row['filename']),
            description = row['df_description']
        )
    return dataset


In [7]:
#More helper functions

import os
import json
import time
import requests
from dvuploader import DVUploader
from pyDataverse.api import NativeApi
from pyDataverse.models import Dataset as PyDataset

# ---------------------------------------------------------------------------
# Checkpoint helpers – persist PIDs to disk so a kernel restart won't lose
# track of already-created datasets.
# ---------------------------------------------------------------------------

def save_pids(pid_file: str, pids: dict) -> None:
    """Write the PID checkpoint file."""
    with open(pid_file, "w") as f:
        json.dump(pids, f, indent=2)
    print(f"  💾 Checkpointed PIDs → {pid_file}")


def load_pids(pid_file: str) -> dict:
    """Read the PID checkpoint file (returns {} if it doesn't exist yet)."""
    if os.path.exists(pid_file):
        with open(pid_file) as f:
            return json.load(f)
    return {}


# ---------------------------------------------------------------------------
# Query Dataverse for files already in the DRAFT version of a dataset.
# Used to filter out already-uploaded files before a retry.
# ---------------------------------------------------------------------------

def get_uploaded_filenames(p_id: str, dataverse_url: str, api_token: str) -> set:
    """
    Return the set of filenames already present in the draft of *p_id*.
    Falls back to an empty set if the request fails so callers can proceed.
    """
    endpoint = (
        f"{dataverse_url.rstrip('/')}/api/datasets/:persistentId"
        f"/versions/:draft/files?persistentId={p_id}"
    )
    headers = {"X-Dataverse-key": api_token}

    try:
        resp = requests.get(endpoint, headers=headers, verify=False, timeout=30)
        resp.raise_for_status()
        return {item["dataFile"]["filename"] for item in resp.json().get("data", [])}
    except Exception as exc:
        print(f"  ⚠️  Could not fetch existing files (will assume none): {exc}")
        return set()


# ---------------------------------------------------------------------------
# Main upload function – replaces dataset.upload()
# ---------------------------------------------------------------------------

def resilient_upload(
    dataset,
    dataverse_name: str,
    n_parallel: int = 1,
    max_retries: int = 5,
    base_delay: int = 60,       # seconds; doubles on each retry
    batch_size: int = 100,      # files per upload batch
) -> str:
    """
    Upload an easyDataverse Dataset with fault-tolerant file handling.

    Differences from dataset.upload():
    ─ dataset.p_id is set immediately after the dataset record is created,
      before any files are transferred.
    ─ Already-uploaded files are detected and skipped before every (re)try.
    ─ Files are uploaded in batches so a 503 only affects one batch.
    ─ Exponential backoff (base_delay × 2^attempt) between retries.

    Parameters
    ----------
    dataset        : easyDataverse Dataset object (with .DATAVERSE_URL / .API_TOKEN)
    dataverse_name : target Dataverse collection alias
    n_parallel     : parallel file streams inside each batch (keep at 1 for safety)
    max_retries    : retry attempts per batch before raising
    base_delay     : initial wait in seconds before the first retry
    batch_size     : number of files per upload call

    Returns
    -------
    str : persistent identifier (PID) of the dataset
    """
    dv_url = str(dataset.DATAVERSE_URL)
    token  = str(dataset.API_TOKEN)
    api    = NativeApi(dv_url, token)

    # ── 1. Create the dataset record (skip if it already has a PID) ──────────
    if dataset.p_id is None:
        print(f"\n📦 Creating dataset: '{dataset.citation.title}'")
        ds_model = PyDataset()
        ds_model.from_json(dataset.dataverse_json())

        resp = api.create_dataset(
            dataverse=dataverse_name,
            metadata=dataset.dataverse_json(),
        )
        resp.raise_for_status()

        # ← store immediately so a later file-upload failure doesn't lose it
        dataset.p_id = resp.json()["data"]["persistentId"]
        print(f"  ✅ Dataset created → {dataset.p_id}")
    else:
        print(f"\n📦 Resuming dataset '{dataset.citation.title}' → {dataset.p_id}")

    p_id = dataset.p_id

    # ── 2. Skip files already present in the draft ───────────────────────────
    already_uploaded = get_uploaded_filenames(p_id, dv_url, token)
    pending = [
        f for f in dataset.files
        if os.path.basename(str(f.filepath)) not in already_uploaded
    ]

    print(f"  📂 {len(already_uploaded)} already uploaded · "
          f"{len(pending)} still to upload out of {len(dataset.files)} total")

    if not pending:
        print("  ✅ Nothing left to upload.")
        return p_id

    # ── 3. Batch upload with exponential-backoff retry ────────────────────────
    batches = [pending[i : i + batch_size] for i in range(0, len(pending), batch_size)]

    for batch_idx, batch in enumerate(batches, start=1):
        print(f"\n  🗂  Batch {batch_idx}/{len(batches)} — {len(batch)} files")

        for attempt in range(1, max_retries + 1):
            try:
                uploader = DVUploader(files=batch)
                uploader.upload(
                    persistent_id=p_id,
                    dataverse_url=dv_url,
                    api_token=token,
                    n_parallel_uploads=n_parallel,
                )
                print(f"  ✅ Batch {batch_idx} complete.")
                break  # ← success; advance to the next batch

            except Exception as exc:
                if attempt == max_retries:
                    print(f"  ❌ Batch {batch_idx} failed after {max_retries} attempts. Re-raising.")
                    raise

                # Re-check what was actually uploaded during the failed attempt
                # so the retry list only contains genuinely missing files.
                uploaded_now = get_uploaded_filenames(p_id, dv_url, token)
                batch = [
                    f for f in batch
                    if os.path.basename(str(f.filepath)) not in uploaded_now
                ]

                if not batch:
                    print(f"  ✅ All files in batch {batch_idx} appear uploaded despite error. Moving on.")
                    break

                delay = base_delay * (2 ** (attempt - 1))
                print(f"  ⚠️  Attempt {attempt} failed ({type(exc).__name__}: {exc})")
                print(f"  ↺  {len(batch)} files remain · retrying in {delay}s …")
                time.sleep(delay)

        # Brief cooldown between batches (not after the last one)
        if batch_idx < len(batches):
            print("  ⏳ Pausing 30s before next batch …")
            time.sleep(30)

    return p_id

## 1. Metadata
**Documentation**
Fields are assigned at the file level in a table. Files are then grouped into datasets according to value in dataset_title field. Some fields are assigned universally in Step 0 to all datasets. <p>
Fields:
1. dataset_title: title of dataset
2. author_name: last, first
3. author_affiliation: ??
4. author_identifier_type: ORCID
5. author_identifier: orcid #
6. contact_name: last, first
7. contact_affiliation: ??
8. contact_email: ??
9. description: dataset description, duplicate for all files in dataset
10. subject: dataset subject
11. keyword: dataset keywords, separated by semicolons
12. depositor: Mika, Katherine
13. license: CC0 1.0
14. language: English??
15. funding_agency: Leon Levy foundation info
16. production_date: date of data production (if known)
17. production_location: Israel
18. filename: filename
19. file_ext: file extension (all lowercase - change in table ahead of deposit if necessary)
20. file_description: OCHRE link with href html markup

#### Internal notes about metadata spreasheets: 
- levy_missing_images_addedphotos = table of photos that have not been uploaded because the image was missing. files are located in "additional_photos" folder
- levy_missing_images_fixedfilename = table of photos that have not been uploaded - due to errors in filenames. files are located in existing photos folders (by year)
- levy_missing_links_fixedlinks = table of photos that have not been uploaded due to errors missing ochre links. files are located in existing photos folder (by year)

*To Do:*
1. Join all tables, dropping rows with identical filenames
2. Add columns for all necessary mta

In [8]:
# import metadata tables
#missing_images_addedphotos = pd.read_csv(source_path + 'photos/remaining_photos/levy_missing_images_addedphotos.csv', index_col=None, low_memory=False)
#missing_images_fixedfiles = pd.read_csv(source_path + 'photos/remaining_photos/levy_missing_images_fixedfilenames.csv', index_col = None)
#missing_links_fixedlinks = pd.read_csv(source_path + 'photos/remaining_photos/levy_missing_links_to_upload.csv', index_col = None)
metadata = pd.read_csv(g_dataverse_inventory_file, index_col = None)
#dir_files = pd.read_csv("/Users/katherinemika/Desktop/curation/leon_levy/illustrations/files/files.csv", index_col = None)

In [9]:
# concatenate tables when importing multiple metadata files
#long_metadata = pd.concat([missing_images_addedphotos, missing_images_fixedfiles, missing_links_fixedlinks])

# drop rows with duplicate 'filename' values, keeping the first occurrence
#metadata = long_metadata.drop_duplicates(subset='filename', keep='first')
#metadata = metadata.drop_duplicates(subset='filename', keep='first')

#to lowercase all chars in filenames col and dir_files
#metadata['filename'] = metadata['filename'].str.lower()
#dir_files['files'] = dir_files['files'].str.lower()

#find missing values in each df
#filename in metadata not in directory:
#metadata['status'] = metadata['filename'].apply(
#    lambda x: 'missing in dir_files' if x not in dir_files['files'].values else ''
#)

#file in directory not listed in metadata sheet:
#dir_files['status'] = dir_files['files'].apply(
#    lambda x: 'missing in metadata' if x not in metadata['filename'].values else ''
#)

#keep only metadata rows that have matching file in dir_rows: 
#metadata_matched = metadata[metadata['status'] == ''].copy()
#metadata_matched.reset_index(drop=True, inplace=True)

# create column for dataset based on year (filename prefix) 
#metadata['year'] = metadata['filename'].str.split('_').str[0]
#metadata['year'] = metadata['year'].str.extract(r'[a-zA-Z](\d{2})')

#metadata.reset_index(drop=True, inplace=True)

# apply the conversion function to the 'year' column
#metadata['year'] = metadata['year'].apply(convert_year)
#metadata['year'] = metadata['year'].fillna(0).astype(int)

#metadata.to_csv("missing_files.csv", index = False)
#dir_files.to_csv('missing_metadata.csv', index = False)

In [10]:
#metadata_matched = metadata_matched.dropna(subset=['filename'])
#metadata_matched.head(-5)

In [11]:
# rename columns
#metadata_matched.rename(columns = {'metadata':'file_description', 'dataset':'dataset_title'}, inplace = True)

# add href wrapper to description link
#metadata_matched['file_description'] = "OCHRE Link: <a href=" + metadata_matched['file_description'] + ">" + metadata_matched['file_description'] + "</a>"

# duplicate dataset_title for prod. date
#metadata['prod_date'] = metadata['dataset_title']
#metadata['prod_date'] = "2004"

# edit dataset_title col
#metadata['dataset_title'] = metadata['dataset_title'].astype(str) + ' Photographs'
#metadata['dataset_title'] = "Ashkelon Surface Photos"

#add dataset_desc column
#metadata['dataset_description'] = "Dataset contains photographs from " + metadata['prod_date'].astype(str) + " excavation."
#metadata_matched['dataset_description'] = "Dataset contains renderings of the excavation site."

#metadata_matched
#metadata_matched.to_csv('files_to_add.csv', index = False)

In [12]:
# split inventory into datasets
# get list of datasets in the full inventory
#change metadata to metadata_matched if using above cells to process metadata tables
g_dataset_titles = list(metadata.dataset_title.unique())

# create inventories of datasets
for title in g_dataset_titles:
    # get inventory of dataset
    g_dataset_inventories[title] = metadata.loc[metadata['dataset_title'] == title]

pprint.pprint(g_dataset_titles)

['Leon Levy Expedition to Ashkelon Dataset: Fieldbooks',
 'Leon Levy Expedition to Ashkelon Dataset: Material Culture Registry',
 'Leon Levy Expedition to Ashkelon Dataset: Field Reports']


In [13]:
for title in g_dataset_titles:
    # get dataset inventory
    dataset_inventory = g_dataset_inventories[title]
    md = create_dataset_metadata(author1_name,
                                 author2_name,
                                 author1_affil,
                                 author2_affil,
                                 dataset_contact,
                                 dataset_contact_email,
                                 subj,
                                 kws,
                                 lang,
                                 funding_agency,
                                 depositor,
                                 geo_coverage,
                                 geo_unit,
                                 title,
                                 dataset_inventory)
    
    g_dataset_metadata[title] = md

pprint.pprint(g_dataset_metadata)

{'Leon Levy Expedition to Ashkelon Dataset: Field Reports': {'author': [{'authorAffiliation': 'Wheaton '
                                                                                              'College',
                                                                         'authorName': 'Master, '
                                                                                       'Daniel '
                                                                                       'M.'},
                                                                        {'authorAffiliation': 'Harvard '
                                                                                              'University',
                                                                         'authorName': 'Stager, '
                                                                                       'Lawrence '
                                                                                       'E.'

In [14]:
#Create dict of DataFrames containing metadata about individual files
for dataset in g_dataset_titles:
    # get dataset metadata 
    dataset_metadata = g_dataset_metadata[dataset]
    # get dataset inventory
    dataset_inventory_df = g_dataset_inventories[dataset]
    # create data file metadata
    g_datafile_metadata[dataset] = create_datafile_metadata(dataset_inventory_df)

# 3. Deposit Data Using EasyDataverse

In [15]:
import sys
import rich
from easyDataverse import Dataverse

In [16]:
dataverse = Dataverse(
    server_url= dv_installation_url,
    api_token= dv_api_key
)

Output()

🎉 Connected to 'https://dvn-cloud-app-2.lib.harvard.edu'

In [17]:
# For each dataset, create a dataset and and save its information

for dataset in g_dataset_titles:
    # initiate dataset
    dv_dataset = dataverse.create_dataset()
    # get metadata 
    dataset_metadata = g_dataset_metadata[dataset]
    # create the dataset
    dv_dataset_info[dataset] = create_dataset(dv_dataset, dataset_metadata)

In [18]:
# add files to datasets
for dataset_title, dataset in dv_dataset_info.items():
    file_inventory = g_datafile_metadata.get(dataset_title)
    # build path for each dataset dir
    dataset_dict = dataset.dataverse_dict()
    citation_fields = dataset_dict.get('datasetVersion', {}).get('metadataBlocks', {}).get('citation', {}).get('fields', [])
    # add files to dataset from inventory using dataset files path
    add_files_to_dataset(dataset, file_inventory, datafiles_path)
    title = dataset.citation.title
    rich.print(f"Added {len(dataset.files)} file to {title}")

Added 343 file to Leon Levy Expedition to Ashkelon Dataset: Fieldbooks

Added 37 file to Leon Levy Expedition to Ashkelon Dataset: Material Culture Registry

Added 428 file to Leon Levy Expedition to Ashkelon Dataset: Field Reports

In [19]:
total_length = sum(len(dataset.files) for dataset_title, dataset in dv_dataset_info.items())
total_length

# compare to length of dataset inventory table

808

In [20]:
#New upload loop - June 2026
# Path for crash-recovery checkpoint
pid_checkpoint_file = os.path.join(source_path, "upload_pids.json")

# Load any PIDs saved from a previous (possibly interrupted) run
collection_pids = load_pids(pid_checkpoint_file)

failed_datasets = {}

for dataset_title, dataset in dv_dataset_info.items():
    print(f"\n{'═' * 60}")
    print(f"  {dataset_title}")
    print(f"{'═' * 60}")

    # ── Restore PID from checkpoint so a kernel restart can resume cleanly ──
    if dataset_title in collection_pids and dataset.p_id is None:
        dataset.p_id = collection_pids[dataset_title]
        print(f"  🔁 Restored PID from checkpoint: {dataset.p_id}")

    dataset.license = dataverse.licenses["CC BY-NC-ND 4.0"]

    try:
        pid = resilient_upload(
            dataset=dataset,
            dataverse_name=dv_collection,
            n_parallel=1,       # 1 is safest against the rate limiter
            max_retries=5,
            base_delay=60,      # 60 → 120 → 240 → 480 → 960 s
            batch_size=100,
        )
        collection_pids[dataset_title] = pid
        save_pids(pid_checkpoint_file, collection_pids)  # ← persist after every dataset
        print(f"\n  🎉 '{dataset_title}' done → {pid}")

    except Exception as exc:
        print(f"\n  ❌ '{dataset_title}' ultimately failed: {exc}")
        failed_datasets[dataset_title] = str(exc)
        # Save progress so you can fix the failure and rerun just this dataset
        save_pids(pid_checkpoint_file, collection_pids)

    # Inter-dataset cooldown (skip after last)
    if dataset_title != list(dv_dataset_info.keys())[-1]:
        print("\n  ⏳ Waiting 5 minutes before next dataset …")
        time.sleep(300)

# ── Final summary ────────────────────────────────────────────────────────────
print(f"\n{'═' * 60}")
print(f"  Succeeded : {len(collection_pids)}")
print(f"  Failed    : {len(failed_datasets)}")
if failed_datasets:
    print(f"  Failed datasets: {list(failed_datasets.keys())}")


════════════════════════════════════════════════════════════
  Leon Levy Expedition to Ashkelon Dataset: Fieldbooks
════════════════════════════════════════════════════════════

📦 Creating dataset: 'Leon Levy Expedition to Ashkelon Dataset: Fieldbooks'
Dataset with pid 'doi:10.7910/DVN/MV7O1B' created.
  ✅ Dataset created → doi:10.7910/DVN/MV7O1B
  📂 0 already uploaded · 343 still to upload out of 343 total

  🗂  Batch 1/4 — 100 files




╭────────────────── DVUploader ───────────────────╮
│ Server: https://dvn-cloud-app-2.lib.harvard.edu │
│ PID: doi:10.7910/DVN/MV7O1B                     │
│ Files: 100                                      │
╰─────────────────────────────────────────────────╯

   🔎 Checking   
  dataset files  
┏━━━━━┳━━━━━━━━━┓
┃ New ┃ Replace ┃
┡━━━━━╇━━━━━━━━━┩
│ 100 │ 0       │
└─────┴─────────┘

🚀 Uploading files

Output()

...

🔎 Checking dataset files                               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃ File                               ┃ Status ┃ Action ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│ 2014_N5_Aja.pdf                    │ New    │ Upload │
│ 2014_Physical_Anthropology_Fox.pdf │ New    │ Upload │
│ 2015_16.39-49-59_Shames.pdf        │ New    │ Upload │
│ 2015_16_Walton.pdf                 │ New    │ Upload │
│ 2015_25.96_Ehrlich.pdf             │ New    │ Upload │
│ 2015_25_Hoffman.pdf                │ New    │ Upload │
│ 2015_47.94_Ehrlich.pdf             │ New    │ Upload │
│ 2015_47_Hoffman.pdf                │ New    │ Upload │
│ 2015_51.73_Dutton.pdf              │ New    │ Upload │
│ 2015_51.74-75_Holtje.pdf           │ New    │ Upload │
│ 2015_51.83_Dutton.pdf              │ New    │ Upload │
│ 2015_51.84_Erickson.pdf            │ New    │ Upload │
│ 2015_51_Wright-Wylie.pdf           │ New    │ Upload │
│ 2015_54_Hoffman.pdf                │ New    │ Upload │
│ 2015_N5.14_Fu.pdf                  │ New    │ Upload │
│ 2015_N5.14_Kalisher.pdf            │ New    │ Upload │
│ 2015_N5.64_Fu.pdf                  │ New    │ Upload │
│ 2015_N5.64_Kalisher.pdf            │ New    │ Upload │
│ 2015_N5_Aja.pdf                    │ New    │ Upload │
│ 2016_51.74_Erickson.pdf            │ New    │ Upload │
│ 2016_51.74-75_Walton.pdf           │ New    │ Upload │
│ 2016_51.83_Dutton.pdf              │ New    │ Upload │
│ 2016_51.84-85_Erickson.pdf         │ New    │ Upload │
│ 2016_51_Wylie.pdf                  │ New    │ Upload │
│ 2016_N5.3_Walton.pdf               │ New    │ Upload │
│ 2016_N5.4_Fu-McCully.pdf           │ New    │ Upload │
│ 2016_N5.24_Hoffman.pdf             │ New    │ Upload │
│ 2016_N5_Aja.pdf                    │ New    │ Upload │
└────────────────────────────────────┴────────┴────────┘

🚀 Uploading files

Output()

✅ Upload complete

  ✅ Batch 5 complete.
  💾 Checkpointed PIDs → /Users/katherinemika/Desktop/curation/leon_levy/summer2026/upload_pids.json

  🎉 'Leon Levy Expedition to Ashkelon Dataset: Field Reports' done → doi:10.7910/DVN/OSFQA7

════════════════════════════════════════════════════════════
  Succeeded : 13
  Failed    : 0


In [18]:
# ── Retry failed datasets after main loop resolves ───────────────────────────

if not failed_datasets:
    print("✅ No failed datasets to retry.")
else:
    print(f"\n{'═' * 60}")
    print(f"  Retrying {len(failed_datasets)} failed dataset(s)")
    print(f"{'═' * 60}")

    # Reload checkpoint in case anything was written during the loop
    collection_pids = load_pids(pid_checkpoint_file)

    retry_succeeded = []
    still_failing   = {}

    for dataset_title in list(failed_datasets.keys()):
        print(f"\n{'─' * 60}")
        print(f"  🔁 Retrying: {dataset_title}")
        print(f"{'─' * 60}")

        dataset = dv_dataset_info[dataset_title]

        # ── Restore PID so resilient_upload skips dataset creation ───────────
        # If PID is in checkpoint the dataset was created but files failed.
        # If PID is NOT in checkpoint the dataset itself never got created —
        # resilient_upload will create it fresh.
        if dataset_title in collection_pids and dataset.p_id is None:
            dataset.p_id = collection_pids[dataset_title]
            print(f"  📌 Restored PID from checkpoint: {dataset.p_id}")
            print(f"     → Dataset already exists; will only upload missing files")
        elif dataset.p_id is not None:
            print(f"  📌 PID already on dataset object: {dataset.p_id}")
        else:
            print(f"  ⚠️  No PID found — dataset will be created from scratch")

        # Restore license (may have been lost if kernel was restarted)
        dataset.license = dataverse.licenses["CC BY-NC-ND 4.0"]

        try:
            pid = resilient_upload(
                dataset=dataset,
                dataverse_name=dv_collection,
                n_parallel=1,
                max_retries=5,
                base_delay=120,     # more generous on retry
                batch_size=25,      # smaller batches on retry
            )
            collection_pids[dataset_title] = pid
            save_pids(pid_checkpoint_file, collection_pids)
            retry_succeeded.append(dataset_title)
            # Remove from failed_datasets now that it succeeded
            del failed_datasets[dataset_title]
            print(f"\n  🎉 '{dataset_title}' retry succeeded → {pid}")

        except Exception as exc:
            print(f"\n  ❌ '{dataset_title}' still failing: {exc}")
            still_failing[dataset_title] = str(exc)

    # ── Retry summary ─────────────────────────────────────────────────────────
    print(f"\n{'═' * 60}")
    print(f"  Retry succeeded : {len(retry_succeeded)}")
    if retry_succeeded:
        for t in retry_succeeded:
            print(f"    • {t}  →  {collection_pids[t]}")
    print(f"  Still failing   : {len(still_failing)}")
    if still_failing:
        for t, err in still_failing.items():
            print(f"    • {t}  —  {err}")
        print(f"\n  💡 For persistent failures, try running the single-dataset")
        print(f"     retry cell with batch_size reduced to 10 or 5.")
    print(f"{'═' * 60}")

✅ No failed datasets to retry.


In [21]:
# ── Targeted retry for a single failed dataset ───────────────────────────────

retry_title = "2022 Photographs"

# Reload checkpoint in case kernel was restarted
collection_pids = load_pids(pid_checkpoint_file)

dataset = dv_dataset_info[retry_title]

# Restore PID from checkpoint so resilient_upload knows not to create a new dataset
if retry_title in collection_pids and dataset.p_id is None:
    dataset.p_id = collection_pids[retry_title]
    print(f"  🔁 Restored PID from checkpoint: {dataset.p_id}")
else:
    print(f"  ℹ️  PID already set on dataset object: {dataset.p_id}")

dataset.license = dataverse.licenses["CC BY-NC-ND 4.0"]

try:
    pid = resilient_upload(
        dataset=dataset,
        dataverse_name=dv_collection,
        n_parallel=1,
        max_retries=5,
        base_delay=60,
        batch_size=50,      # ← reduced from 100; "Too much data" suggests smaller chunks help
    )
    collection_pids[retry_title] = pid
    save_pids(pid_checkpoint_file, collection_pids)
    print(f"\n  🎉 '{retry_title}' done → {pid}")

except Exception as exc:
    print(f"\n  ❌ Still failing: {exc}")
    print("  💡 Try reducing batch_size further (e.g. 10) or check your VPN connection.")

  ℹ️  PID already set on dataset object: doi:10.7910/DVN/HBCIZR

📦 Resuming dataset '2022 Photographs' → doi:10.7910/DVN/HBCIZR
  📂 800 already uploaded · 146 still to upload out of 946 total

  🗂  Batch 1/3 — 50 files




╭───────────── DVUploader ──────────────╮
│ Server: https://dataverse.harvard.edu │
│ PID: doi:10.7910/DVN/HBCIZR           │
│ Files: 50                             │
╰───────────────────────────────────────╯

🔎 Checking dataset files          
┏━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃ File          ┃ Status ┃ Action ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│ A22_34917.jpg │ New    │ Upload │
│ A22_34918.jpg │ New    │ Upload │
│ A22_34919.jpg │ New    │ Upload │
│ A22_34920.jpg │ New    │ Upload │
│ A22_34921.jpg │ New    │ Upload │
│ A22_34922.jpg │ New    │ Upload │
│ A22_34923.jpg │ New    │ Upload │
│ A22_34924.jpg │ New    │ Upload │
│ A22_34925.jpg │ New    │ Upload │
│ A22_34926.jpg │ New    │ Upload │
│ A22_34927.jpg │ New    │ Upload │
│ A22_34928.jpg │ New    │ Upload │
│ A22_34929.jpg │ New    │ Upload │
│ A22_34930.jpg │ New    │ Upload │
│ A22_34931.jpg │ New    │ Upload │
│ A22_34932.jpg │ New    │ Upload │
│ A22_34933.jpg │ New    │ Upload │
│ A22_34934.jpg │ New    │ Upload │
│ A22_34935.jpg │ New    │ Upload │
│ A22_34936.jpg │ New    │ Upload │
│ A22_34937.jpg │ New    │ Upload │
│ A22_34938.jpg │ New    │ Upload │
│ A22_34939.jpg │ New    │ Upload │
│ A22_34940.jpg │ New    │ Upload │
│ A22_34941.jpg │ New    │ Upload │
│ A22_34942.jpg │ New    │ Upload │
│ A22_34943.jpg │ New    │ Upload │
│ A22_34944.jpg │ New    │ Upload │
│ A22_34945.jpg │ New    │ Upload │
│ A22_34946.jpg │ New    │ Upload │
│ A22_34947.jpg │ New    │ Upload │
│ A22_34948.jpg │ New    │ Upload │
│ A22_34949.jpg │ New    │ Upload │
│ A22_34950.jpg │ New    │ Upload │
│ A22_34951.jpg │ New    │ Upload │
│ A22_34952.jpg │ New    │ Upload │
│ A22_34953.jpg │ New    │ Upload │
│ A22_34954.jpg │ New    │ Upload │
│ A22_34955.jpg │ New    │ Upload │
│ A22_34956.jpg │ New    │ Upload │
│ A22_34957.jpg │ New    │ Upload │
│ A22_34958.jpg │ New    │ Upload │
│ A22_34959.jpg │ New    │ Upload │
│ A22_34960.jpg │ New    │ Upload │
│ A22_34961.jpg │ New    │ Upload │
│ A22_34962.jpg │ New    │ Upload │
│ A22_34963.jpg │ New    │ Upload │
│ A22_34964.jpg │ New    │ Upload │
│ A22_34965.jpg │ New    │ Upload │
│ A22_34966.jpg │ New    │ Upload │
└───────────────┴────────┴────────┘

🚀 Uploading files

Output()

  ⚠️  Attempt 1 failed (LocalProtocolError: Too much data for declared Content-Length)
  ↺  50 files remain · retrying in 60s …


KeyboardInterrupt: 

### Set user permissions for all sub-collections

Necessary step because permissions do not inherit! 

In [19]:
def get_subcollections(parent_collection_id, dataverse_url, api_token):
    """
    Retrieve all sub-collections within a parent Dataverse collection.
    Filters to only 'dataverse' type items (excludes datasets).

    Parameters
    ----------
    parent_collection_id : str
        Alias or ID of the parent collection (e.g. 'leon_levy')
    dataverse_url : str
        Base URL of the Dataverse installation
    api_token : str
        Dataverse API token

    Returns
    -------
    list of dict : each dict contains 'id' and 'title' for one sub-collection
    """
    endpoint = f"{dataverse_url.rstrip('/')}/api/dataverses/{parent_collection_id}/contents"
    headers  = {"X-Dataverse-key": api_token}

    print(f"📋 Fetching contents of '{parent_collection_id}'...")

    try:
        resp = requests.get(endpoint, headers=headers, verify=False, timeout=30)
        resp.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"  ❌ HTTP error fetching contents: {e}")
        print(f"  Response body: {resp.text[:500]}")
        return []
    except Exception as e:
        print(f"  ❌ Unexpected error fetching contents: {e}")
        return []

    contents = resp.json().get("data", [])

    # Filter to sub-collections only (type == "dataverse")
    # Note: the API response only returns 'id' and 'title', no 'alias'
    subcollections = [
        {
            "id"    : item["id"],
            "title" : item.get("title", "Untitled"),
        }
        for item in contents
        if item.get("type") == "dataverse"
    ]

    print(f"  ✅ Found {len(subcollections)} sub-collection(s) "
          f"(skipped {len(contents) - len(subcollections)} non-dataverse items)")

    return subcollections


def assign_role(collection_id, assignee, role, dataverse_url, api_token):
    """
    Assign a role to a user for a given Dataverse collection.

    Parameters
    ----------
    collection_id : str or int
        Numeric ID of the target collection
    assignee : str
        Dataverse username prefixed with '@' (e.g. '@dmmaster')
    role : str
        Role alias to assign (e.g. 'admin', 'curator', 'editor')
    dataverse_url : str
        Base URL of the Dataverse installation
    api_token : str
        Dataverse API token

    Returns
    -------
    bool : True if assignment succeeded, False otherwise
    """
    endpoint = f"{dataverse_url.rstrip('/')}/api/dataverses/{collection_id}/assignments"
    headers  = {
        "X-Dataverse-key" : api_token,
        "Content-Type"    : "application/json",
    }
    payload = {
        "assignee" : assignee,
        "role"     : role,
    }

    try:
        resp = requests.post(
            endpoint,
            headers=headers,
            json=payload,
            verify=False,
            timeout=30,
        )
        resp.raise_for_status()
        print(f"    ✅ Assigned '{role}' to '{assignee}' on collection '{collection_id}'")
        return True

    except requests.exceptions.HTTPError as e:
        print(f"    ❌ HTTP error assigning role on '{collection_id}': {e}")
        print(f"       Response body: {resp.text[:300]}")
        return False

    except Exception as e:
        print(f"    ❌ Unexpected error assigning role on '{collection_id}': {e}")
        return False


def assign_role_to_all_subcollections(
    parent_collection_id,
    assignee,
    role,
    dataverse_url,
    api_token,
    include_parent=False,
):
    """
    Full pipeline: fetch all sub-collections under parent_collection_id,
    then assign the given role to assignee for every one of them.

    Parameters
    ----------
    parent_collection_id : str
        Alias or ID of the parent collection (e.g. 'leon_levy')
    assignee : str
        Dataverse username prefixed with '@' (e.g. '@dmmaster')
    role : str
        Role alias to assign (e.g. 'admin', 'curator')
    dataverse_url : str
        Base URL of the Dataverse installation
    api_token : str
        Dataverse API token
    include_parent : bool
        If True, also assign the role on the parent collection itself.
        Defaults to False.

    Returns
    -------
    dict : {'succeeded': [...], 'failed': [...]}
    """
    print(f"\n{'═' * 60}")
    print(f"  Role assignment pipeline")
    print(f"  Parent collection : {parent_collection_id}")
    print(f"  Assignee          : {assignee}")
    print(f"  Role              : {role}")
    print(f"{'═' * 60}\n")

    # ── Step 1: get sub-collections ───────────────────────────────────────────
    subcollections = get_subcollections(parent_collection_id, dataverse_url, api_token)

    if not subcollections:
        print("  ⚠️  No sub-collections found. Exiting.")
        return {"succeeded": [], "failed": []}

    print(f"\n  Sub-collections to process:")
    for sc in subcollections:
        print(f"    • [{sc['id']}]  {sc['title']}")

    # Optionally prepend the parent itself
    targets = subcollections.copy()
    if include_parent:
        targets.insert(0, {
            "id"    : parent_collection_id,
            "title" : "(parent collection)",
        })

    # ── Step 2: assign role on every target ───────────────────────────────────
    print(f"\n  Assigning roles...")

    succeeded = []
    failed    = []

    for sc in targets:
        target_id = sc["id"]
        ok = assign_role(target_id, assignee, role, dataverse_url, api_token)

        if ok:
            succeeded.append({"id": target_id, "title": sc["title"]})
        else:
            failed.append({"id": target_id, "title": sc["title"]})

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'═' * 60}")
    print(f"  ✅ Succeeded : {len(succeeded)}")
    if succeeded:
        for s in succeeded:
            print(f"    • [{s['id']}]  {s['title']}")
    print(f"  ❌ Failed    : {len(failed)}")
    if failed:
        for f in failed:
            print(f"    • [{f['id']}]  {f['title']}")
    print(f"{'═' * 60}\n")

    return {"succeeded": succeeded, "failed": failed}

In [20]:
# ── Run it ────────────────────────────────────────────────────────────────────
results = assign_role_to_all_subcollections(
    parent_collection_id = dv_collection,  # parent collection alias
    assignee             = "@dmmaster",
    role                 = "admin",
    dataverse_url        = dv_installation_url,
    api_token            = dv_api_key,
    include_parent       = False,        # set True to also assign on leon_levy itself
)


════════════════════════════════════════════════════════════
  Role assignment pipeline
  Parent collection : levy_photos
  Assignee          : @dmmaster
  Role              : admin
════════════════════════════════════════════════════════════

📋 Fetching contents of 'levy_photos'...
  ✅ Found 28 sub-collection(s) (skipped 33 non-dataverse items)

  Sub-collections to process:
    • [4040415]  2012 Photographs
    • [4549804]  1995 Photographs
    • [4550541]  1996 Photographs
    • [4550543]  1997 Photographs
    • [4550544]  1998 Photographs
    • [4550546]  1999 Photographs
    • [4550547]  2000 Photographs
    • [4638448]  1993 Photographs
    • [4880636]  1985 Photographs
    • [4880646]  1986 Photographs
    • [4883659]  1987 Photographs
    • [4887813]  1988 Photographs
    • [4892626]  1989 Photographs
    • [4916701]  1990 Photographs
    • [4929239]  1991 Photographs
    • [4930201]  1992 Photographs
    • [4936295]  1994 Photographs
    • [4938341]  2004 Photographs
    • [4

### Request file DOIs for dataset

In [19]:
import requests
import json

In [66]:
#Note! In API guide, requests may follow ID or persistentId format. All endpoints can use either ID or persistentId according to the
#method described in the guide (typically ID) or the method used here: persistentId.

#Note 2! Version may or may not need to be specified, dpending on the endpoint. For published datasets, the default is "latestVersion."
#For drafts, you typically will need to indicate ":draft" as seen below. 

# Reuses the credentials loaded in the global-variables cell.
# NOTE: this previously hard-coded SERVER_URL to production
# (https://dataverse.harvard.edu) even when deposits ran against staging.
# It now follows DATAVERSE_TARGET; override here if you need to harvest DOIs
# from a different installation than you deposited to.
SERVER_URL = dv_installation_url
VERSION = ":draft"
API_KEY = dv_api_key
headers = {'X-Dataverse-key': API_KEY}
photos_dois = pd.DataFrame()

In [22]:
def get_file_dois(
    dataset_dict,
    dataverse_url,
    api_token,
    version=":latest",
    output_csv_path=None,
    public_base_url="https://dataverse.harvard.edu",
):
    """
    Retrieve file-level DOIs for multiple datasets and build a
    consolidated DataFrame with direct download links.

    Parameters
    ----------
    dataset_dict : dict
        Keys are dataset titles (str), values are dataset DOIs (str).
        e.g. {"2022 Photographs": "doi:10.7910/DVN/28B3QA", ...}
    dataverse_url : str
        Base URL of the Dataverse installation (used for API calls).
    api_token : str
        Dataverse API token.
    version : str
        Dataset version to query. Defaults to ':latest'.
        Other options: ':latest-published', '1.0', '2.1', etc.
    output_csv_path : str or None
        Full path to write the output CSV. If None, CSV is not written.
    public_base_url : str
        Base URL used to construct direct download links.
        Defaults to 'https://dataverse.harvard.edu'.

    Returns
    -------
    pd.DataFrame
        Columns: filename, file_doi, dataset_name, dataset_doi,
                 direct_download_link
    """

    headers = {"X-Dataverse-key": api_token}
    all_rows = []
    failed   = {}

    print(f"📋 Fetching file DOIs for {len(dataset_dict)} dataset(s)...\n")

    for dataset_name, dataset_doi in dataset_dict.items():
        print(f"  📂 {dataset_name}  ({dataset_doi})")

        url = (
            f"{dataverse_url.rstrip('/')}/api/datasets/:persistentId"
            f"/versions/{version}/files?persistentId={dataset_doi}"
        )

        try:
            response = requests.get(url, headers=headers, verify=False, timeout=30)
            response.raise_for_status()
        except requests.exceptions.HTTPError as e:
            print(f"    ❌ HTTP error: {e}")
            print(f"       Response: {response.text[:300]}")
            failed[dataset_name] = str(e)
            continue
        except Exception as e:
            print(f"    ❌ Unexpected error: {e}")
            failed[dataset_name] = str(e)
            continue

        file_list = response.json().get("data", [])

        if not file_list:
            print(f"    ⚠️  No files returned for this dataset")
            continue

        dataset_rows = []
        missing_doi  = 0

        for file_entry in file_list:
            datafile = file_entry.get("dataFile", {})
            filename = datafile.get("filename")
            file_doi = datafile.get("persistentId")

            if not file_doi:
                missing_doi += 1

            # Build direct download link from file DOI
            direct_link = (
                f"{public_base_url.rstrip('/')}"
                f"/api/access/datafile/:persistentId"
                f"?persistentId={file_doi}"
                if file_doi else None
            )

            dataset_rows.append({
                "filename"             : filename,
                "file_doi"             : file_doi,
                "dataset_name"         : dataset_name,
                "dataset_doi"          : dataset_doi,
                "direct_download_link" : direct_link,
            })

        all_rows.extend(dataset_rows)

        print(f"    ✅ {len(dataset_rows)} files retrieved", end="")
        if missing_doi:
            print(f"  ⚠️  {missing_doi} file(s) had no DOI", end="")
        print()

    # ── Build final DataFrame ─────────────────────────────────────────────────
    df = pd.DataFrame(all_rows, columns=[
        "filename",
        "file_doi",
        "dataset_name",
        "dataset_doi",
        "direct_download_link",
    ])

    # ── Write CSV ─────────────────────────────────────────────────────────────
    if output_csv_path:
        df.to_csv(output_csv_path, index=False)
        print(f"\n  💾 CSV written → {output_csv_path}")

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'═' * 60}")
    print(f"  Total files      : {len(df)}")
    print(f"  Datasets queried : {len(dataset_dict)}")
    print(f"  Failed datasets  : {len(failed)}")
    if failed:
        for name, err in failed.items():
            print(f"    • {name}  —  {err}")
    print(f"{'═' * 60}")

    return df


In [24]:
# ── Run it ────────────────────────────────────────────────────────────────────

# Build the input dict from your existing collection_pids checkpoint,
# or define it manually
dataset_doi_dict = {
    "2019 Photographs": "doi:10.7910/DVN/GC7VPA",
    "2022 Photographs": "doi:10.7910/DVN/I61NUB",
    "Leon Levy Expedition to Ashkelon Dataset: Videos": "doi:10.7910/DVN/PUPH8R",
    "Pottery Registry: Dataset 1": "doi:10.7910/DVN/AIZG6H",
    "Pottery Registry: Dataset 2": "doi:10.7910/DVN/WHN3EV",
    "Pottery Readings: Dataset 1": "doi:10.7910/DVN/2DRF2T",
    "Pottery Readings: Dataset 2": "doi:10.7910/DVN/LPMXIY",
    "Pottery Readings: Dataset 3": "doi:10.7910/DVN/FQ4IV2",
    "Pottery Readings: Dataset 4": "doi:10.7910/DVN/GGMWB6",
    "Pottery Readings: Dataset 5": "doi:10.7910/DVN/T5LVID",
    "Pottery Readings: Dataset 6": "doi:10.7910/DVN/KKNVNP",
    "Leon Levy Expedition to Ashkelon Dataset: Fieldbooks": "doi:10.7910/DVN/MV7O1B",
    "Leon Levy Expedition to Ashkelon Dataset: Material Culture Registry": "doi:10.7910/DVN/OD9HLS",
    "Leon Levy Expedition to Ashkelon Dataset: Field Reports": "doi:10.7910/DVN/OSFQA7"
}

# Or build it automatically from your collection_pids checkpoint:
# dataset_doi_dict = collection_pids  # keys = titles, values = DOIs

file_dois_df = get_file_dois(
    dataset_dict    = dataset_doi_dict,
    dataverse_url   = dv_installation_url,
    api_token       = dv_api_key,
    version         = ":latest",
    output_csv_path = os.path.join(source_path, "file_dois.csv"),
    public_base_url = "https://dataverse.harvard.edu",
)

file_dois_df.head()

📋 Fetching file DOIs for 14 dataset(s)...

  📂 2019 Photographs  (doi:10.7910/DVN/GC7VPA)
    ✅ 197 files retrieved
  📂 2022 Photographs  (doi:10.7910/DVN/I61NUB)
    ✅ 946 files retrieved
  📂 Leon Levy Expedition to Ashkelon Dataset: Videos  (doi:10.7910/DVN/PUPH8R)
    ✅ 405 files retrieved
  📂 Pottery Registry: Dataset 1  (doi:10.7910/DVN/AIZG6H)
    ✅ 899 files retrieved
  📂 Pottery Registry: Dataset 2  (doi:10.7910/DVN/WHN3EV)
    ✅ 500 files retrieved
  📂 Pottery Readings: Dataset 1  (doi:10.7910/DVN/2DRF2T)
    ✅ 898 files retrieved
  📂 Pottery Readings: Dataset 2  (doi:10.7910/DVN/LPMXIY)
    ✅ 901 files retrieved
  📂 Pottery Readings: Dataset 3  (doi:10.7910/DVN/FQ4IV2)
    ✅ 900 files retrieved
  📂 Pottery Readings: Dataset 4  (doi:10.7910/DVN/GGMWB6)
    ✅ 900 files retrieved
  📂 Pottery Readings: Dataset 5  (doi:10.7910/DVN/T5LVID)
    ✅ 900 files retrieved
  📂 Pottery Readings: Dataset 6  (doi:10.7910/DVN/KKNVNP)
    ✅ 425 files retrieved
  📂 Leon Levy Expedition to Ashkel

,filename,file_doi,dataset_name,dataset_doi,direct_download_link
0,A19_33914.jpg,doi:10.7910/DVN/GC7VPA/XT6P9S,2019 Photographs,doi:10.7910/DVN/GC7VPA,https://dataverse.harvard.edu/api/access/dataf...
1,A19_33915.jpg,doi:10.7910/DVN/GC7VPA/4W0A11,2019 Photographs,doi:10.7910/DVN/GC7VPA,https://dataverse.harvard.edu/api/access/dataf...
2,A19_33916.jpg,doi:10.7910/DVN/GC7VPA/CFVAXV,2019 Photographs,doi:10.7910/DVN/GC7VPA,https://dataverse.harvard.edu/api/access/dataf...
3,A19_33917.jpg,doi:10.7910/DVN/GC7VPA/TN3TY4,2019 Photographs,doi:10.7910/DVN/GC7VPA,https://dataverse.harvard.edu/api/access/dataf...
4,A19_33918.jpg,doi:10.7910/DVN/GC7VPA/JLMI8H,2019 Photographs,doi:10.7910/DVN/GC7VPA,https://dataverse.harvard.edu/api/access/dataf...


In [121]:
#get all files in SINGLE dataset
PID = "doi:10.7910/DVN/28B3QA" #assign this manually if only requesting DOIs for one dataset. 
url = "{}/api/datasets/:persistentId/versions/{}/files?persistentId={}".format(SERVER_URL, VERSION, PID)
response = requests.get(url, headers = headers)
datasets = response.json()
dataset_metadata = datasets['data']

#parse response to get dois
#metadata[0][0]['dataFile']['filename']

# List to accumulate rows
rows = []

# Loop through the nested list
for file in dataset_metadata:
    datafile = file.get('dataFile', {})
    filename = datafile.get('filename')
    doi = datafile.get('persistentId')
    #Append as a dict
    rows.append({'filename': filename, 'DOI': doi})

# Create DataFrame all at once & append to illustrations_dois
df = pd.DataFrame(rows)
photos_dois = pd.concat([photos_dois, df], ignore_index=True)

In [122]:
photos_dois

,filename,DOI
0,A85_184B.jpg,doi:10.7910/DVN/0Z3FC1/QGJTAX
1,A85_296b.jpg,doi:10.7910/DVN/0Z3FC1/X1K79T
2,A85_315b.jpg,doi:10.7910/DVN/0Z3FC1/RT9A95
3,A85_315c.jpg,doi:10.7910/DVN/0Z3FC1/POZ965
4,A85_315d.jpg,doi:10.7910/DVN/0Z3FC1/FPF7EG
...,...,...
2743,A19_33909.jpg,doi:10.7910/DVN/28B3QA/5LM3T1
2744,A19_33910.jpg,doi:10.7910/DVN/28B3QA/BHVFDI
2745,A19_33911.jpg,doi:10.7910/DVN/28B3QA/DHOJI4
2746,A19_33912.jpg,doi:10.7910/DVN/28B3QA/QO3OK2


In [124]:
photos_dois.to_csv("/Users/katherinemika/Desktop/curation/leon_levy/illustrations/photos_files_dois.csv", index = False)

In [21]:
#get all PIDs in a collection 
url = "{}/api/dataverses/{}/contents".format(SERVER_URL, dv_collection)
response = requests.get(url, headers = headers)
datasets = response.json()

In [22]:
dataset_dois = []
for item in datasets['data']:
    dataset_dois.append(f"doi:10.7910/{item['identifier']}")

dataset_dois

['doi:10.7910/DVN/9S3JTN',
 'doi:10.7910/DVN/FUJ1AC',
 'doi:10.7910/DVN/7LRAEQ',
 'doi:10.7910/DVN/AEUYOC',
 'doi:10.7910/DVN/Y23OH9',
 'doi:10.7910/DVN/ZK9VLW',
 'doi:10.7910/DVN/X1LU8T',
 'doi:10.7910/DVN/UCS72H',
 'doi:10.7910/DVN/ZGXU2M',
 'doi:10.7910/DVN/JGIF2H',
 'doi:10.7910/DVN/AD8LMX',
 'doi:10.7910/DVN/FUOGIZ',
 'doi:10.7910/DVN/2Y3PF3',
 'doi:10.7910/DVN/TC8GZQ',
 'doi:10.7910/DVN/BL4GMW',
 'doi:10.7910/DVN/7PPFLN',
 'doi:10.7910/DVN/JZFYR4',
 'doi:10.7910/DVN/IAHTDW',
 'doi:10.7910/DVN/HG8MYJ',
 'doi:10.7910/DVN/KEEUSH',
 'doi:10.7910/DVN/NOAFYQ',
 'doi:10.7910/DVN/8SGKFH',
 'doi:10.7910/DVN/BNLWJE',
 'doi:10.7910/DVN/VMC6XQ',
 'doi:10.7910/DVN/B8TRKI',
 'doi:10.7910/DVN/UIQOJU',
 'doi:10.7910/DVN/UGMBOQ',
 'doi:10.7910/DVN/9TICC3',
 'doi:10.7910/DVN/GBWVB3',
 'doi:10.7910/DVN/EHLYBV',
 'doi:10.7910/DVN/MQUIJX',
 'doi:10.7910/DVN/1ICKOL',
 'doi:10.7910/DVN/ZKIYFK',
 'doi:10.7910/DVN/QWSOUK',
 'doi:10.7910/DVN/IG9XJT',
 'doi:10.7910/DVN/AXEE57',
 'doi:10.7910/DVN/O86S7R',
 

In [34]:
#get files from dataset dois
dataset_metadata = []
for doi in dataset_dois:
    url = "{}/api/datasets/:persistentId/versions/{}/files?persistentId={}".format(SERVER_URL, VERSION, doi)
    response = requests.get(url, headers = headers)
    dataset = response.json()
    dataset_metadata.append(dataset['data'])

In [40]:
#parse JSON output to access metadata fields for each file. 
dataset_metadata[0][0]['dataFile']['filename']

# List to accumulate rows
rows = []

# Loop through the nested list
for dataset in dataset_metadata:
    for file in dataset:
        datafile = file.get('dataFile', {})
        filename = datafile.get('filename')
        doi = datafile.get('persistentId')
        # Append as a dict
        rows.append({'filename': filename, 'DOI': doi})

# Create DataFrame all at once
df = pd.DataFrame(rows)

In [43]:
illustrations_dois = df
illustrations_dois

,filename,DOI
0,dwgid0001.jpg,doi:10.7910/DVN/9S3JTN/WYCYHI
1,dwgid0003.jpg,doi:10.7910/DVN/9S3JTN/EVW4JB
2,dwgid0004.jpg,doi:10.7910/DVN/9S3JTN/FK0F20
3,dwgid0005.jpg,doi:10.7910/DVN/9S3JTN/GB0MSO
4,dwgid0006.jpg,doi:10.7910/DVN/9S3JTN/TTDBY9
...,...,...
26372,dwgid29339.jpg,doi:10.7910/DVN/DWMWNC/NYTLYF
26373,dwgid29340.jpg,doi:10.7910/DVN/DWMWNC/LSOSZA
26374,dwgid29341.jpg,doi:10.7910/DVN/DWMWNC/T6EB1G
26375,dwgid29342.jpg,doi:10.7910/DVN/DWMWNC/Z26COQ


In [45]:
illustrations_dois.to_csv("/Users/katherinemika/Desktop/curation/leon_levy/illustrations/illustrations_dois.csv", index = False)

## Add resolved images/files to existing datasets

In [23]:
#import new spreadsheet
#make column for year based on filename
#get all dataset DOIs and add to DF as new column (dataset_DOI)
#for row in df: create file matedata payload and upload to dataset in dataset_doi column

In [100]:
july25 = pd.read_csv("/Users/katherinemika/Desktop/curation/leon_levy/photos/remaining_photos/july_25.csv")

# create column for dataset based on year (filename prefix) 
july25['year'] = july25['filename'].str.split('_').str[0]
july25['year'] = july25['year'].str.extract(r'[a-zA-Z](\d{2})')

july25.reset_index(drop=True, inplace=True)

# function to convert two-digit year to four-digit year is from early in script
july25['year'] = july25['year'].apply(convert_year)
july25['year'] = july25['year'].fillna(0).astype(int)

july25

,ochre_metadata,filename,Dataset,year
0,http://pi.lib.uchicago.edu/1001/org/ochre/b119...,A99_12624.jpg,doi:10.7910/DVN/K4QL7B,1999


In [101]:
files_by_dataset = {}

# Group by dataset
for dataset, group in july25.groupby('Dataset'):
    files_list = [
        {
            'filepath': os.path.join(datafiles_path, row['filename']),
            'description': f"<a href='{row['ochre_metadata']}'>{row['ochre_metadata']}</a>"
        }
        for _, row in group.iterrows()
    ]
    files_by_dataset[dataset] = files_list

# The files_by_dataset dictionary now contains lists of files for each dataset
files_by_dataset['doi:10.7910/DVN/K4QL7B']

[{'filepath': '/Users/katherinemika/Desktop/curation/leon_levy/photos/july25/A99_12624.jpg',
  'description': "<a href='http://pi.lib.uchicago.edu/1001/org/ochre/b1197a83-05a6-592b-ef8f-9a00d81fd3d0'>http://pi.lib.uchicago.edu/1001/org/ochre/b1197a83-05a6-592b-ef8f-9a00d81fd3d0</a>"}]

In [102]:
unique_datasets = july25['Dataset'].unique().tolist()
unique_datasets

['doi:10.7910/DVN/K4QL7B']

In [103]:
for doi in unique_datasets:
    print("Dataset: ",doi, files_by_dataset[doi])

Dataset:  doi:10.7910/DVN/K4QL7B [{'filepath': '/Users/katherinemika/Desktop/curation/leon_levy/photos/july25/A99_12624.jpg', 'description': "<a href='http://pi.lib.uchicago.edu/1001/org/ochre/b1197a83-05a6-592b-ef8f-9a00d81fd3d0'>http://pi.lib.uchicago.edu/1001/org/ochre/b1197a83-05a6-592b-ef8f-9a00d81fd3d0</a>"}]


## Old code I don't want to delete

In [ ]:
#inital loop I used to deposit datasets from easydataverse to HDV
# deposit datasets to repo (all) - be sure to enable FILE DOIS if necessary. Code is in "dataverse-scripts" folder if necessary.
collection_pids = {}
for dataset_title, dataset in list(dv_dataset_info.items()): #add eg. "[1:]" if not first pass. This is for subsequent uploads when retrying
    #set license
    dataset.license = dataverse.licenses["CC BY-NC-ND 4.0"]
    #upload datasets and files
    pid = dataset.upload(dataverse_name = dv_collection, n_parallel=2)
    collection_pids[dataset_title] = pid
    
    # Wait 5 minutes before uploading the next dataset
    print(f"Uploaded '{dataset_title}', waiting 5 minutes before next upload...")
    time.sleep(300)  # 300 seconds = 5 minutes

## Troubleshooting

There's somthing funny about the metadata for the 1997 dataset that is causing the files to fail to register. Seems like the "file metadata" is the problem. Ideas for fixing
- save metadata to csv file to investigate
- reconfirm that all files in directory are not corrupted (or something)
- use the dvuploader to test file upload with no added metadata (eg. without ochre or tags)

**A:** There were some issues with filenames, so I flagged them. 

In [ ]:
for dataset_title, dataset in dv_dataset_info.items():
    file_inventory = g_datafile_metadata.get(dataset_title)
    # build path for each dataset dir
    dataset_dict = dataset.dataverse_dict()
    citation_fields = dataset_dict.get('datasetVersion', {}).get('metadataBlocks', {}).get('citation', {}).get('fields', [])
    for field in citation_fields:
        if field.get('typeName') == 'productionDate':
            # Extract and print the productionDate value
            production_date = field.get('value')
            dataset_files_path = os.path.join(datafiles_path, str(production_date))
            break
    # add files to dataset from inventory using dataset files path
    curate.add_files_to_dataset(dataset, file_inventory, dataset_files_path)
    title = dataset.citation.title
    rich.print(f"Added {len(dataset.files)} file to {title}")

In [17]:
#test for one dataset
#dv_dataset_info['1991 Photographs']

test_file_inventory = g_datafile_metadata.get('1991 Photographs')
dataset_dict = dv_dataset_info['1991 Photographs'].dataverse_dict()
citation_fields = dataset_dict.get('datasetVersion', {}).get('metadataBlocks', {}).get('citation', {}).get('fields', [])
for field in citation_fields:
    if field.get('typeName') == 'productionDate':
        # Extract and print the productionDate value
        production_date = field.get('value')
        dataset_files_path = os.path.join(datafiles_path, str(production_date))
        break
curate.add_files_to_dataset(dv_dataset_info['1991 Photographs'], test_file_inventory, dataset_files_path)
title = dv_dataset_info['1991 Photographs'].citation.title
rich.print(f"Added {len(dv_dataset_info['1991 Photographs'].files)} file to {title}")

Added 50 file to 1991 Photographs

In [18]:
test_file_inventory = g_datafile_metadata.get('1991 Photographs')
test_file_inventory

,filename,df_description
0,A91_4093.jpg,OCHRE Link: <a href=http://pi.lib.uchicago.edu...
1,A91_4209.jpg,OCHRE Link: <a href=http://pi.lib.uchicago.edu...
2,A91_4233a1.jpg,OCHRE Link: <a href=http://pi.lib.uchicago.edu...
3,A91_4273.jpg,OCHRE Link: <a href=http://pi.lib.uchicago.edu...
4,A91_4304.jpg,OCHRE Link: <a href=http://pi.lib.uchicago.edu...
5,A91_4305.jpg,OCHRE Link: <a href=http://pi.lib.uchicago.edu...
6,A91_4306.jpg,OCHRE Link: <a href=http://pi.lib.uchicago.edu...
7,A91_4307.jpg,OCHRE Link: <a href=http://pi.lib.uchicago.edu...
8,A91_4308.jpg,OCHRE Link: <a href=http://pi.lib.uchicago.edu...
9,A91_4309.jpg,OCHRE Link: <a href=http://pi.lib.uchicago.edu...


In [18]:
collection_pids = {}
pid = dv_dataset_info['1991 Photographs'].upload(dataverse_name = dv_collection)
collection_pids['1991 Photographs'] = pid

Dataset with pid 'doi:10.7910/DVN/SVC2FI' created.




╭───────────── DVUploader ──────────────╮
│ Server: https://dataverse.harvard.edu │
│ PID: doi:10.7910/DVN/SVC2FI           │
│ Files: 50                             │
╰───────────────────────────────────────╯

Output()

🔎 Checking dataset files            
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃ File            ┃ Status ┃ Action ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│ A91_4093.jpg    │ New    │ Upload │
│ A91_4209.jpg    │ New    │ Upload │
│ A91_4233a1.jpg  │ New    │ Upload │
│ A91_4273.jpg    │ New    │ Upload │
│ A91_4304.jpg    │ New    │ Upload │
│ A91_4305.jpg    │ New    │ Upload │
│ A91_4306.jpg    │ New    │ Upload │
│ A91_4307.jpg    │ New    │ Upload │
│ A91_4308.jpg    │ New    │ Upload │
│ A91_4309.jpg    │ New    │ Upload │
│ A91_4310.jpg    │ New    │ Upload │
│ A91_4311.jpg    │ New    │ Upload │
│ A91_4312.jpg    │ New    │ Upload │
│ A91_4312b.jpg   │ New    │ Upload │
│ A91_4313.jpg    │ New    │ Upload │
│ A91_4314.jpg    │ New    │ Upload │
│ A91_4315.jpg    │ New    │ Upload │
│ A91_4316.jpg    │ New    │ Upload │
│ A91_4317.jpg    │ New    │ Upload │
│ A91_4318.jpg    │ New    │ Upload │
│ A91_4319.jpg    │ New    │ Upload │
│ A91_4320.jpg    │ New    │ Upload │
│ A91_4321.jpg    │ New    │ Upload │
│ A91_4322.jpg    │ New    │ Upload │
│ A91_4323.jpg    │ New    │ Upload │
│ A91_4324.jpg    │ New    │ Upload │
│ A91_3929-p.jpg  │ New    │ Upload │
│ A91_3942b.jpg   │ New    │ Upload │
│ A91_3986-p.jpg  │ New    │ Upload │
│ A91_3993-p.jpg  │ New    │ Upload │
│ A91_3994-p.jpg  │ New    │ Upload │
│ A91_3995-p.jpg  │ New    │ Upload │
│ A91_3996-p.jpg  │ New    │ Upload │
│ A91_4038-p.jpg  │ New    │ Upload │
│ A91_4052.jpg    │ New    │ Upload │
│ A91_4076-p.jpg  │ New    │ Upload │
│ A91_4099-p.jpg  │ New    │ Upload │
│ A91_4151-p.jpg  │ New    │ Upload │
│ A91_4153-p.jpg  │ New    │ Upload │
│ A91_4161-p.jpg  │ New    │ Upload │
│ A91_4171b-p.jpg │ New    │ Upload │
│ A91_4172-p.jpg  │ New    │ Upload │
│ A91_4205-p.jpg  │ New    │ Upload │
│ A91_4232.jpg    │ New    │ Upload │
│ A91_4234a.jpg   │ New    │ Upload │
│ A91_4234b.jpg   │ New    │ Upload │
│ A91_4237.jpg    │ New    │ Upload │
│ A91_4238.jpg    │ New    │ Upload │
│ A91_4240.jpg    │ New    │ Upload │
│ A91_4987.jpg    │ New    │ Upload │
└─────────────────┴────────┴────────┘

🚀 Uploading files

Output()

✅ Upload complete

╭─ Dataset URL ───────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  🎉 https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/SVC2FI                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
#next steps: redetect file type; other QA?; publish? 

In [19]:
# get all datasetIDs
headers = {'X-Dataverse-key': dv_api_key}
url = f"{dv_installation_url}/api/dataverses/{dv_collection}/contents"
response = requests.get(url, headers=headers)

dataset_ids = []
for r in response.json()['data']:
    dataset_ids.append(r['id'])

In [20]:
dataset_ids

[11194521]

In [21]:
# get all fileIDs for redetect
file_ids = []

for dataset in dataset_ids: 
    headers = {'X-Dataverse-key': dv_api_key}
    url = f"{dv_installation_url}/api/datasets/{dataset}/versions/:draft/files"
    response = requests.get(url, headers = headers)
    for r in response.json()['data']:
        file_ids.append(r['dataFile']['id'])

In [22]:
file_ids

[11194568,
 11194537,
 11194570,
 11194551,
 11194527,
 11194536,
 11194535,
 11194572,
 11194565,
 11194566,
 11194545,
 11194531,
 11194557,
 11194553,
 11194548,
 11194555,
 11194560,
 11194563,
 11194528,
 11194529,
 11194558,
 11194573,
 11194546,
 11194571,
 11194562,
 11194564,
 11194549,
 11194547,
 11194554,
 11194524,
 11194569,
 11194543,
 11194561,
 11194552,
 11194556,
 11194539,
 11194532,
 11194550,
 11194567,
 11194525,
 11194538,
 11194544,
 11194533,
 11194559,
 11194540,
 11194530,
 11194526,
 11194542,
 11194534,
 11194541]

In [23]:
# redetect files
for id in file_ids:
    headers = {'X-Dataverse-key': dv_api_key}
    url = f"{dv_installation_url}/api/files/{id}/redetect"
    response = requests.post(url, headers=headers)
    print(response.json())
    time.sleep(2)

{'status': 'OK', 'data': {'dryRun': False, 'oldContentType': 'text/plain', 'newContentType': 'image/jpeg'}}
{'status': 'OK', 'data': {'dryRun': False, 'oldContentType': 'text/plain', 'newContentType': 'image/jpeg'}}
{'status': 'OK', 'data': {'dryRun': False, 'oldContentType': 'text/plain', 'newContentType': 'image/jpeg'}}
{'status': 'OK', 'data': {'dryRun': False, 'oldContentType': 'text/plain', 'newContentType': 'image/jpeg'}}
{'status': 'OK', 'data': {'dryRun': False, 'oldContentType': 'text/plain', 'newContentType': 'image/jpeg'}}
{'status': 'OK', 'data': {'dryRun': False, 'oldContentType': 'text/plain', 'newContentType': 'image/jpeg'}}
{'status': 'OK', 'data': {'dryRun': False, 'oldContentType': 'text/plain', 'newContentType': 'image/jpeg'}}
{'status': 'OK', 'data': {'dryRun': False, 'oldContentType': 'text/plain', 'newContentType': 'image/jpeg'}}
{'status': 'OK', 'data': {'dryRun': False, 'oldContentType': 'text/plain', 'newContentType': 'image/jpeg'}}
{'status': 'OK', 'data': {'d